# Generative Model Accuracy Analysis

Compare predicted vs. reference molecular properties (e.g. `eMBD`, `HLgap`) across the generative models produced by successive active-learning iterations, and across different experiment/training setups (active-loop, active-loop-non-opt, vanilla-diffusion, trajectories).

**What this notebook does**

- Loads per-model, per-target evaluation CSVs (`eMBD.csv`, `HLgap.csv`) and the SAScore tiering file (`sascore.txt`).
- Plots error / relative-error distributions per model as box plots and violin plots, for a single target, two targets, and all targets together.
- Compares molecule counts and "valid molecule" counts (accurate + high SAScore tier) across experiment setups.
- Produces a top-N (lowest relative error) molecule comparison across experiments and targets.

**How to use it**
- Fill in `DATA_ROOT` in the *Configuration* cell below — every other path in the notebook is derived from it.
- Run *Configuration*, *Setup*, and *Shared helpers* first. After that, sections are largely independent and can be run in any order — but see the **Notebook map** below, since a few sections reuse the `df_all` produced by the section right above them.

**Expected data layout** (relative to each experiment folder, e.g. `data_full_dataset/`)
```
<experiment>/evaluation/target<N>/model<M>/results/eMBD.csv       # molecule_name, calculated, reference, error
<experiment>/evaluation/target<N>/model<M>/results/HLgap.csv      # same columns, different property
<experiment>/evaluation/target<N>/model<M>/results/sascore.txt    # molecule_name, score, tier
```

**Notebook map**

| Section | Needs | Produces / reused by |
|---|---|---|
| Single target | `DATA_FULL_DATASET` | standalone |
| Two Targets | `DATA_EMBD_JMP` | standalone |
| All Targets | `DATA_EMBD_JMP` | standalone |
| Final | `DATA_VANILLA_FULL_DATASET` | `df_all` used by *Summary statistics* below it |
| Trajectory | `DATA_STRAIGHT_TRAJ`, `DATA_TRIANGLE_TRAJ` | overwrites `df_all` - rerun *Final* if you need it back |
| Summary statistics | `df_all` from *Final* or *Trajectory* (whichever ran last) | standalone report |
| Molecule counts | `DATA_EMBD_JMP`, `DATA_EMBD_NON_OPT`, `DATA_VANILLA` | standalone |
| "Valid molecule" counts | same three experiments | standalone |
| Frequencies (top-40 tiers) | same three experiments | standalone |
| Top-N molecule comparison | `data_full_dataset`, `data_vanilla` | standalone |

*Single target / Two Targets / All Targets / Final / Trajectory* all follow the same pattern (load → combine → plot) and share the `load_targets(...)` helper defined below, so if you need a new comparison, copy one of those cells and change its arguments rather than writing the loop from scratch.

## Configuration
Set `DATA_ROOT` to the folder that contains all the `data_*` experiment sub-folders used below. Everything else in this notebook is derived from it.

In [ ]:
from pathlib import Path

# Root directory containing all experiment sub-folders (data_full_dataset, data_embd_jmp, ...).
DATA_ROOT = Path("/path/to/your/data")  # <-- SET THIS

# Individual experiment folders referenced throughout this notebook.
DATA_FULL_DATASET = DATA_ROOT / "data_full_dataset"
DATA_EMBD_JMP = DATA_ROOT / "data_embd_jmp"
DATA_EMBD_NON_OPT = DATA_ROOT / "data_embd_non_opt"
DATA_VANILLA = DATA_ROOT / "data_vanilla"
DATA_VANILLA_FULL_DATASET = DATA_ROOT / "data_vanilla_full_dataset"
DATA_STRAIGHT_TRAJ = DATA_ROOT / "data_straight_traj"
DATA_TRIANGLE_TRAJ = DATA_ROOT / "data_triangle_traj"


## Setup: install dependencies
Run once per environment.

In [ ]:
%pip install -q --upgrade seaborn matplotlib scipy


## Imports

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


## Shared helper: load a property across models/targets
Every "load CSV per model/target, compute error, combine" loop in this notebook follows the same shape, so it's factored out here once. Later sections just call `load_targets(...)` with different arguments instead of repeating the loop.

In [ ]:
def load_targets(target_folders, csv_name="eMBD.csv", columns=None,
                  subsample_n=None, top_n_by_rel_error=None, random_state=42):
    """
    Load a per-model results CSV across one or more (folder, target_label) pairs and combine
    into a single DataFrame with "model" and "target" columns.

    Parameters
    ----------
    target_folders : list[tuple[Path, str]]
        (folder, target_label) pairs, e.g. [(DATA_EMBD_JMP / "evaluation/target0", "target0")].
    csv_name : str
        File name to read from each model's "results" folder (e.g. "eMBD.csv" or "HLgap.csv").
    columns : list[str], optional
        Column names for the (headerless, tab-separated) CSV. Defaults to the standard
        ["molecule_name", "calculated_property", "reference_property", "error"].
    subsample_n : int, optional
        If set, randomly subsample to at most this many molecules per model/target.
    top_n_by_rel_error : int, optional
        If set, instead keep only the N molecules with the lowest |relative error| per
        model/target (adds a "relative_error" column). Mutually exclusive with subsample_n.

    Returns
    -------
    pd.DataFrame
    """
    if columns is None:
        columns = ["molecule_name", "calculated_property", "reference_property", "error"]

    data = []
    for folder, target_name in target_folders:
        for model_name in os.listdir(folder):
            model_path = os.path.join(folder, model_name)
            if not os.path.isdir(model_path):
                continue

            csv_path = os.path.join(model_path, "results", csv_name)
            if not os.path.exists(csv_path):
                continue

            df = pd.read_csv(csv_path, header=None, names=columns, sep="\t")
            df["model"] = model_name
            df["target"] = target_name

            if "error" not in df.columns or df["error"].isnull().all():
                df["error"] = df["calculated_property"] - df["reference_property"]

            if top_n_by_rel_error is not None:
                df["relative_error"] = (df["calculated_property"] - df["reference_property"]) / df["reference_property"]
                df = df.loc[df["relative_error"].abs().sort_values(ascending=True).index].head(top_n_by_rel_error)
            elif subsample_n is not None and len(df) > subsample_n:
                df = df.sample(n=subsample_n, random_state=random_state)

            data.append(df)

    df_all = pd.concat(data, ignore_index=True)
    df_all["model"] = pd.Categorical(df_all["model"], categories=sorted(df_all["model"].unique()), ordered=True)
    return df_all


## Single target: `target3`, active-loop full dataset
The manual, unrolled version of the load shown above — a good starting point if you're adapting this notebook to a new file layout. Loads `eMBD.csv` for every model under `data_full_dataset/evaluation/target3` and compares calculated vs. reference values, and their error, per model.

In [ ]:
target_folder = DATA_FULL_DATASET / "evaluation/target3"
columns = ["molecule_name", "calculated_property", "reference_property", "error"]
data = []

# Loop through each model folder
for model_name in os.listdir(target_folder):
    model_path = os.path.join(target_folder, model_name)
    if not os.path.isdir(model_path):
        continue  # skip files
    csv_path = os.path.join(model_path, "results/eMBD.csv")

    df = pd.read_csv(csv_path, header=None, names=columns, sep="\t")
    df["model"] = model_name
    data.append(df)

# Combine all model data
df_all = pd.concat(data, ignore_index=True)


In [ ]:
# Sort models alphabetically
sorted_models = sorted(df_all["model"].unique())

# Prepare data for boxplot in sorted order
box_data = [df_all[df_all["model"] == m]["calculated_property"].values for m in sorted_models]

plt.figure(figsize=(10, 6))
plt.boxplot(box_data, labels=sorted_models)
plt.xticks(rotation=45)
plt.ylabel("Calculated Property (embd)")
plt.xlabel("Generative Model")
plt.title("Distribution of Calculated Properties per Generative Model")

# Horizontal line at reference, if all molecules share the same reference value
if df_all["reference_property"].nunique() == 1:
    ref_value = df_all["reference_property"].iloc[0]
    plt.axhline(ref_value, color="red", linestyle="--", label=f"Reference = {ref_value:.3f}")
    plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
box_data = [df_all[df_all["model"] == m]["error"].values for m in sorted_models]

plt.figure(figsize=(10, 6))
plt.boxplot(box_data, labels=sorted_models)
plt.xticks(rotation=45)
plt.ylabel("Error (Calculated - Reference)")
plt.xlabel("Generative Model")
plt.title("Error Distribution per Generative Model")
plt.axhline(0, color="red", linestyle="--", label="Reference")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
df_all["model"] = pd.Categorical(df_all["model"], categories=sorted(df_all["model"].unique()), ordered=True)

plt.figure(figsize=(12, 6))
sns.violinplot(data=df_all, x="model", y="error", palette="Set3", inner="quartile")
plt.axhline(0, color="red", linestyle="--", label="Reference")
plt.xticks(rotation=45)
plt.ylabel("Error (Calculated - Reference)")
plt.xlabel("Generative Model")
plt.title("Error Distribution per Generative Model (Violin Plot)")
plt.legend()
plt.tight_layout()
plt.show()


## Two Targets
Same idea using `load_targets`: `eMBD.csv` for `target0` and `target1` of `data_embd_jmp`, subsampled to 40 molecules per model, compared side by side.

In [ ]:
df_all = load_targets(
    target_folders=[
        (DATA_EMBD_JMP / "evaluation/target0", "target0"),
        (DATA_EMBD_JMP / "evaluation/target1", "target1"),
    ],
    csv_name="eMBD.csv",
    subsample_n=40,
)

plt.figure(figsize=(14, 6))
sns.violinplot(data=df_all, x="model", y="error", hue="target", palette="Set2", split=True, inner="quartile")
plt.axhline(0, color="red", linestyle="--", label="Reference")
plt.xticks(rotation=45)
plt.ylabel("Error (Calculated - Reference)")
plt.xlabel("Generative Model")
plt.title("Error Distribution per Generative Model Across Targets")
plt.legend(title="Target")
plt.tight_layout()
plt.show()


## All Targets
Same idea extended to 4 targets. **Note:** `target2`/`target3` below intentionally reuse the `target0`/`target1` folders as a placeholder — point them at real `target2`/`target3` folders once available.

In [ ]:
df_all = load_targets(
    target_folders=[
        (DATA_EMBD_JMP / "evaluation/target0", "target0"),
        (DATA_EMBD_JMP / "evaluation/target1", "target1"),
        (DATA_EMBD_JMP / "evaluation/target0", "target2"),  # placeholder - replace with real target2 folder
        (DATA_EMBD_JMP / "evaluation/target1", "target3"),  # placeholder - replace with real target3 folder
    ],
    csv_name="eMBD.csv",
    subsample_n=40,
)


In [ ]:
plt.figure(figsize=(16, 6))
sns.violinplot(data=df_all, x="model", y="error", hue="target", palette="Set2", split=True, inner="quartile")
sns.stripplot(data=df_all, x="model", y="error", color="black", size=3, jitter=True)
plt.axhline(0, color="red", linestyle="--", label="Reference")
plt.xticks(rotation=45)
plt.ylabel("Error (Calculated - Reference)")
plt.xlabel("Generative Model")
plt.title("Error Distribution per Generative Model Across 4 Targets")
plt.legend(title="Target")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(14, 6))
sns.violinplot(data=df_all, x="model", y="error", palette="Set3", inner="quartile", scale="width")
sns.stripplot(data=df_all, x="model", y="error", color="black", size=3, jitter=True)
plt.axhline(0, color="red", linestyle="--", label="Reference")
plt.xticks(rotation=45)
plt.ylabel("Error (Calculated - Reference)")
plt.xlabel("Generative Model")
plt.title("Combined Error Distribution per Model (Subsampled to 40 molecules each)")
plt.legend()
plt.tight_layout()
plt.show()


## Final
Using `HLgap.csv` from `data_vanilla_full_dataset` across `target0`-`target2`, take the top-40 molecules (lowest relative error) per model, then compare relative-error distributions. *(This section's `df_all` is reused by the Summary statistics section below.)*

In [ ]:
df_all = load_targets(
    target_folders=[
        (DATA_VANILLA_FULL_DATASET / "evaluation/target0", "target0"),
        (DATA_VANILLA_FULL_DATASET / "evaluation/target1", "target1"),
        (DATA_VANILLA_FULL_DATASET / "evaluation/target2", "target2"),
        # (DATA_VANILLA_FULL_DATASET / "evaluation/target3", "target3"),
    ],
    csv_name="HLgap.csv",
    top_n_by_rel_error=40,
)


In [ ]:
plt.figure(figsize=(16, 6))
sns.violinplot(data=df_all, x="model", y="relative_error", hue="target", palette="Set2", split=True, inner="quartile")
sns.stripplot(data=df_all, x="model", y="relative_error", hue="target", palette="Set2", size=3, jitter=True, dodge=True, legend=False)
plt.axhline(0, color="red", linestyle="--", label="Perfect Prediction")
plt.axhline(0.05, color="blue", linestyle="--", label="+0.05 Relative Error")
plt.axhline(-0.05, color="blue", linestyle="--", label="-0.05 Relative Error")
plt.xticks(rotation=45)
plt.ylabel("Relative Error ((Calculated - Reference)/Reference)")
plt.xlabel("Generative Model")
plt.title("Error Distribution per Generative Model Across 4 Targets")
plt.legend(title="Target", loc="lower right")
plt.tight_layout()
plt.ylim(-1.5, 1.5)
plt.show()


In [ ]:
# Mean absolute relative error per model/target, with std-dev error bars
mean_errors = df_all.groupby(["model", "target"])["relative_error"].mean().abs().reset_index()
std_errors = df_all.groupby(["model", "target"])["relative_error"].std().abs().reset_index()

plt.figure(figsize=(12, 6))
sns.lineplot(data=mean_errors, x="model", y="relative_error", hue="target", palette="Set1", marker="o")
plt.errorbar(x=mean_errors["model"], y=mean_errors["relative_error"], yerr=std_errors["relative_error"],
             fmt="none", color="black", capsize=5)
plt.xticks(rotation=45)
plt.ylabel("Relative Error ((Calculated - Reference)/Reference)")
plt.xlabel("Generative Model")
plt.title("Mean Error per Generative Model Across 4 Targets")
plt.legend(title="Target", loc="lower right")
plt.tight_layout()
plt.ylim(0, 0.01)
plt.show()


In [ ]:
# Box plot of relative error per model, one box per target, sorted by median error across targets
median_errors = df_all.groupby("model")["relative_error"].median().sort_values()
sorted_models = median_errors.index.tolist()

plt.figure(figsize=(12, 6))
sns.boxplot(x="model", y="relative_error", hue="target", data=df_all[df_all["model"].isin(sorted_models)], order=sorted_models)
plt.xticks(rotation=45)
plt.ylabel("Relative Error ((Calculated - Reference)/Reference)")
plt.xlabel("Generative Model")
plt.title("Relative Error Distribution per Generative Model Across 3 Targets")
plt.legend(title="Target", loc="upper right")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(14, 6))
sns.violinplot(data=df_all, x="model", y="relative_error", palette="Set3", inner="quartile", scale="width")
sns.stripplot(data=df_all, x="model", y="relative_error", color="black", size=3, jitter=True)
plt.axhline(0, color="red", linestyle="--", label="Perfect Prediction")
plt.axhline(0.05, color="blue", linestyle="--", label="+0.05 Relative Error")
plt.axhline(-0.05, color="blue", linestyle="--", label="-0.05 Relative Error")
plt.ylim(-0.3, 0.3)
plt.xticks(rotation=45)
plt.ylabel("Relative Error ((Calculated - Reference)/Reference)")
plt.xlabel("Generative Model")
plt.title("Relative Error Distribution per Model (Subsampled to 40 molecules each)")
plt.legend()
plt.tight_layout()
plt.show()


**Reading the plots above:** the shape and width of the violins stay fairly constant across `model0` → `model8`, suggesting predictive precision doesn't degrade over the active-learning loop. There's a slight tightening trend up to around `model7`, consistent with the models slowly gaining stability/confidence as the loop progresses.

## Trajectory
Compare `eMBD` relative error for the 4th target across two trajectory experiments (straight vs. triangle). **Note:** this overwrites `df_all` — rerun the *Final* section above if you need that data back for Summary statistics.

In [ ]:
df_all = load_targets(
    target_folders=[
        (DATA_STRAIGHT_TRAJ / "evaluation/target4", "Straight_Trajectory"),
        (DATA_TRIANGLE_TRAJ / "evaluation/target4", "Triangle_Trajectory"),
    ],
    csv_name="eMBD.csv",
    top_n_by_rel_error=40,
)


In [ ]:
plt.figure(figsize=(16, 6))
sns.violinplot(data=df_all, x="model", y="relative_error", hue="target", palette="Set2", split=True, inner="quartile")
sns.stripplot(data=df_all, x="model", y="relative_error", hue="target", palette="Set2", size=3, jitter=True, dodge=True, legend=False)
plt.axhline(0, color="red", linestyle="--", label="Perfect Prediction")
plt.axhline(0.05, color="blue", linestyle="--", label="+0.05 Relative Error")
plt.axhline(-0.05, color="blue", linestyle="--", label="-0.05 Relative Error")
plt.xticks(rotation=45)
plt.ylabel("Relative Error ((Calculated - Reference)/Reference)")
plt.xlabel("Generative Model")
plt.title("eMBD Error Distribution per Generative Model for the 4th Target")
plt.legend(title="Target", loc="lower right")
plt.tight_layout()
plt.ylim(-0.2, 0.2)
plt.show()


## Summary statistics per model
Per-model median/mean/MAD and the percentage of predictions within ±5% / ±10% relative error. Operates on whichever `df_all` is currently loaded (the *Final* section, by default, unless you ran *Trajectory* more recently).

In [ ]:
df_all = df_all.copy()
# Clip extreme relative errors so summary stats aren't dominated by a few outliers
df_all["relative_error_clipped"] = df_all["relative_error"].clip(-2, 2)

def mad(series):
    return (series - series.median()).abs().median()

grouped = df_all.groupby("model")["relative_error_clipped"]
summary = grouped.agg(
    count="count",
    median="median",
    mean="mean",
    std="std",
    mad=lambda x: mad(x),
    pct_within_0_05=lambda x: (x.abs() <= 0.05).mean() * 100,
    pct_within_0_1=lambda x: (x.abs() <= 0.1).mean() * 100,
).sort_index()

pd.options.display.float_format = "{:.4f}".format
print(summary)

# To also check whether error trends with model index (model0, model1, ...), or plot
# median / pct-within-threshold as bar charts, extend this cell using `summary` above.


**Reading the summary stats:** across the active-loop models, relative error tends to stay tight and roughly unbiased, sometimes with a small upward drift in median error (i.e. the models slightly overpredict as the loop progresses), while the proportion of accurate predictions (within ±5%) can still improve — consistent with the active-learning loop shifting toward a different, harder region of molecule space.

## Molecule counts across experiment setups
How many molecules does each model/target/experiment combination actually produce (`eMBD.csv` row counts)? This section (and the two below it) define their own small file-reading helpers, since each needs slightly different parsing/validation (row counts vs. SAScore tiers vs. merged accuracy+tier filtering) — they're kept close to the code that uses them rather than merged into `load_targets` above, which assumes a different (already-known) column layout.

In [ ]:
folders = {
    "active-loop": DATA_EMBD_JMP,
    "active-loop-non-opt": DATA_EMBD_NON_OPT,
    "vanilla-diffusion": DATA_VANILLA,
}

targets = range(4)   # target0..target3
models = range(9)    # model0..model8
labels = list(folders.keys())

def safe_count_csv_rows(csv_path):
    """Return number of rows in a CSV; 0 if missing, empty, or unparsable."""
    if not csv_path.exists():
        return 0
    try:
        df = pd.read_csv(csv_path, low_memory=False)
        return len(df)
    except pd.errors.EmptyDataError:
        return 0
    except Exception as e:
        print(f"Warning: couldn't read {csv_path!s}: {e}")
        return 0

def safe_count_sas_tiers(txt_path, tiers_to_count={1, 2, 3}):
    """Count sascore.txt rows whose tier (3rd column) is in tiers_to_count; 0 if missing/empty."""
    if not txt_path.exists():
        return 0
    try:
        df = pd.read_csv(txt_path, sep=r"\s+", header=None, engine="python", comment="#", skip_blank_lines=True)
        if df.shape[0] == 0:
            return 0
        if df.shape[1] < 3:
            print(f"Warning: {txt_path!s} has fewer than 3 columns (has {df.shape[1]}). Counting 0.")
            return 0
        tier_col = df.iloc[:, 2].astype(str).str.extract(r"(\d+)")[0]
        tier_col = pd.to_numeric(tier_col, errors="coerce")
        return int(tier_col.isin(list(tiers_to_count)).sum())
    except pd.errors.EmptyDataError:
        return 0
    except Exception as e:
        print(f"Warning: couldn't parse {txt_path!s}: {e}")
        return 0

# ---------- eMBD.csv row counts ----------
results_embd = {label: [[0] * len(models) for _ in targets] for label in labels}

for label in labels:
    dirname = folders[label]
    for t in targets:
        for m in models:
            csv_path = dirname / f"evaluation/target{t}/model{m}/results/eMBD.csv"
            results_embd[label][t][m] = safe_count_csv_rows(csv_path)

# ---------- Plot ----------
x = np.arange(len(models))
width = 0.25

fig, axs = plt.subplots(2, 2, figsize=(14, 10))
axs = axs.flatten()
for t in targets:
    ax = axs[t]
    for i, label in enumerate(labels):
        offsets = (i - 1) * width
        ax.bar(x + offsets, results_embd[label][t], width=width, label=label)
    ax.set_title(f"Target {t} - # molecules in eMBD.csv (rows)")
    ax.set_xlabel("Model index (0-8)")
    ax.set_xticks(x)
    ax.set_xticklabels([str(m) for m in models])
    ax.set_ylabel("# molecules (rows)")
    ax.legend()
    ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()


### SAScore tier counts
Same comparison, but counting molecules by SAScore tier (0-3) instead of raw row count. Reuses `safe_count_sas_tiers` defined above.

In [ ]:
for tier in (3, 2, 1, 0):
    results_sas = {label: [[0] * len(models) for _ in targets] for label in labels}
    for label in labels:
        dirname = folders[label]
        for t in targets:
            for m in models:
                txt_path = dirname / f"evaluation/target{t}/model{m}/results/sascore.txt"
                results_sas[label][t][m] = safe_count_sas_tiers(txt_path, tiers_to_count={tier})

    fig, axs = plt.subplots(2, 2, figsize=(14, 10))
    axs = axs.flatten()
    for t in targets:
        ax = axs[t]
        for i, label in enumerate(labels):
            offsets = (i - 1) * width
            ax.bar(x + offsets, results_sas[label][t], width=width, label=label)
        ax.set_title(f"Target {t} - # molecules in sascore.txt (tier = {tier})")
        ax.set_xlabel("Model index (0-8)")
        ax.set_xticks(x)
        ax.set_xticklabels([str(m) for m in models])
        ax.set_ylabel(f"# molecules (tier {tier})")
        ax.legend()
        ax.set_ylim(0, 80)
    plt.tight_layout()
    plt.show()


## "Valid molecule" counts: tier_3 + within ±5% relative error
For each experiment/target/model, merge `eMBD.csv` with `sascore.txt` and count molecules that are both high SAScore tier (`tier_3`) and within 5% relative error of the reference. Uses its own `safe_read_csv` since, unlike the sections above, both files here are read with the default comma-separated parser — double check this matches your actual file format.

In [ ]:
def safe_read_csv(path):
    """Read a CSV with no header; return None if missing, empty, or unreadable."""
    if not path.exists() or path.stat().st_size == 0:
        return None
    try:
        df = pd.read_csv(path, header=None)
        return None if df.empty else df
    except Exception:
        return None

def compute_valid_molecules(path_embd, path_sas):
    """Count molecules with tier_3 SAScore and |relative error| <= 5%."""
    df_embd = safe_read_csv(path_embd)
    df_sas = safe_read_csv(path_sas)
    if df_embd is None or df_sas is None:
        return 0

    embd_names = df_embd.iloc[:, 0].astype(str)
    col2 = pd.to_numeric(df_embd.iloc[:, 2], errors="coerce")
    col3 = pd.to_numeric(df_embd.iloc[:, 3], errors="coerce")
    rel_err = (col3 / col2) * 100
    df_embd_clean = pd.DataFrame({"name": embd_names, "rel_err": rel_err}).dropna()

    sas_names = df_sas.iloc[:, 0].astype(str)
    sas_tier = df_sas.iloc[:, 2].astype(str)
    df_sas_clean = pd.DataFrame({"name": sas_names, "tier": sas_tier})

    merged = df_embd_clean.merge(df_sas_clean, on="name", how="inner")
    merged = merged[merged["tier"] == "tier_3"]
    merged = merged[merged["rel_err"].abs() <= 5]
    return len(merged)


experiments = {
    "data_embd_jmp": DATA_EMBD_JMP,
    "data_embd_non_opt": DATA_EMBD_NON_OPT,
    "data_vanilla": DATA_VANILLA,
}
targets = [0, 1, 2, 3]
models = range(8)

results = {exp: {t: {m: 0 for m in models} for t in targets} for exp in experiments}

for exp, exp_dir in experiments.items():
    for t in targets:
        for m in models:
            embd_path = exp_dir / f"evaluation/target{t}/model{m}/results/eMBD.csv"
            sas_path = exp_dir / f"evaluation/target{t}/model{m}/results/sascore.txt"
            results[exp][t][m] = compute_valid_molecules(embd_path, sas_path)

for t in targets:
    plt.figure(figsize=(12, 6))
    for exp in experiments:
        y = [results[exp][t][m] for m in models]
        plt.plot(models, y, label=exp, marker="o")

    plt.title(f"Target {t}: Tier_3 + |Relative Error| <= 5%")
    plt.xlabel("Model")
    plt.ylabel("Number of valid molecules")
    plt.ylim(0, 50)  # fixed y-axis max - adjust if your data exceeds this
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()


## Frequencies: top-40 molecules by tier acceptance
For each model, take the top-40 molecules by lowest relative error, then split by SAScore tier group ("good" = tier_2/tier_3 vs. "bad" = tier_0/tier_1) and by relative-error acceptance threshold. Uses its own `safe_read`, which — unlike the section above — parses both files as whitespace-separated; adjust `sep` if your `eMBD.csv` is actually comma- or tab-delimited.

In [ ]:
def safe_read(path, sep=r"\s+"):
    if not path.exists() or path.stat().st_size == 0:
        return None
    try:
        df = pd.read_csv(path, header=None, sep=sep, engine="python")
        return None if df.empty else df
    except Exception:
        return None

def compute_top40_stats(path_embd, path_sas, rel_err_threshold, tier_accept, top_k=40):
    """
    Returns (n_top40, n_accurate, n_final) where:
      - n_top40: number of molecules in the top-k by lowest |relative error|
      - n_accurate: of those, how many are within rel_err_threshold
      - n_final: of the accurate ones, how many fall in an accepted SAScore tier
    """
    df_embd = safe_read(path_embd)
    df_sas = safe_read(path_sas)

    if df_embd is None or df_embd.shape[1] < 3:
        return 0, 0, 0

    names = df_embd.iloc[:, 0].astype(str).str.strip().str.replace(r"\s+", "", regex=True)
    pred = pd.to_numeric(df_embd.iloc[:, 1], errors="coerce")
    ref = pd.to_numeric(df_embd.iloc[:, 2], errors="coerce")
    rel_err = ((pred - ref).abs() / ref.abs() * 100).replace([float("inf"), -float("inf")], pd.NA)

    df = pd.DataFrame({"name": names, "rel_err": rel_err}).dropna()

    df_top40 = df.loc[df["rel_err"].abs().sort_values().index].head(top_k)
    top40_names = set(df_top40["name"].values)

    df_acc = df_top40[df_top40["rel_err"] <= rel_err_threshold]
    acc_names = set(df_acc["name"].values)

    if df_sas is None or df_sas.shape[1] < 3:
        return len(df_top40), len(df_acc), 0

    sas_names = df_sas.iloc[:, 0].astype(str).str.strip().str.replace(r"\s+", "", regex=True)
    sas_tier = df_sas.iloc[:, 2].astype(str)
    df_sas_clean = pd.DataFrame({"name": sas_names, "tier": sas_tier}).drop_duplicates(subset=["name"])

    df_acc_tiers = df_sas_clean[df_sas_clean["name"].isin(acc_names)]
    df_final = df_acc_tiers[df_acc_tiers["tier"].isin(tier_accept)]

    return len(df_top40), len(df_acc), len(df_final)


In [ ]:
experiments = ["data_embd_jmp", "data_embd_non_opt", "data_vanilla"]
exp_dirs = [DATA_EMBD_JMP, DATA_EMBD_NON_OPT, DATA_VANILLA]
exp_labels = ["active-loop", "active-loop-non-opt", "vanilla-diffusion"]
targets = [0, 1, 2, 3]
models = range(9)
REL_ERR_THRESHOLD = 5  # +/-5%
TIER_ACCEPT_G = ["tier_2", "tier_3"]  # "good" tiers
TIER_ACCEPT_B = ["tier_0", "tier_1"]  # "bad" tiers

results_final_g = {label: {t: [0] * len(models) for t in targets} for label in exp_labels}
results_final_b = {label: {t: [0] * len(models) for t in targets} for label in exp_labels}

for exp_dir, label in zip(exp_dirs, exp_labels):
    for t in targets:
        for m in models:
            path_embd = exp_dir / f"evaluation/target{t}/model{m}/results/eMBD.csv"
            path_sas = exp_dir / f"evaluation/target{t}/model{m}/results/sascore.txt"

            _, _, final_count_g = compute_top40_stats(path_embd, path_sas, REL_ERR_THRESHOLD, TIER_ACCEPT_G)
            results_final_g[label][t][m] = final_count_g

            _, _, final_count_b = compute_top40_stats(path_embd, path_sas, REL_ERR_THRESHOLD, TIER_ACCEPT_B)
            results_final_b[label][t][m] = final_count_b


In [ ]:
x = list(models)
colors = ["tab:blue", "tab:orange", "tab:green"]
target_names = ["Target(eMBD:-1.4, atNUM:50)", "Target(eMBD:-1.9, atNUM:60)",
                "Target(eMBD:-2.4, atNUM:75)", "Target(eMBD:-2.9, atNUM:90)"]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()
max_y = 40

for idx, t in enumerate(targets):
    ax = axes[idx]

    for i, label in enumerate(exp_labels):
        ax.plot(x, results_final_g[label][t], color=colors[i], linestyle="-", linewidth=2,
                label=f"{label} (tier 2+3)")
    for i, label in enumerate(exp_labels):
        ax.plot(x, results_final_b[label][t], color=colors[i], linestyle="--", linewidth=2,
                label=f"{label} (tier 0+1)")

    ax.set_title(target_names[t])
    ax.set_xlabel("Model")
    ax.set_ylabel("Final Count")
    ax.set_xticks(x)
    ax.set_ylim(0, max_y * 1.1)
    ax.grid(True, alpha=0.3)

axes[-1].legend(loc="lower right")
plt.tight_layout()
plt.show()


## Top-N molecule comparison across experiments
For `data_full_dataset` and `data_vanilla`, take the top/bottom-N molecules by relative error per model/target and overlay them on the violin plots, colored by SAScore tier.

In [ ]:
from matplotlib.lines import Line2D

top = True  # True = lowest abs rel error, False = highest
n = 20      # number of molecules to sample per model

experiments = ["data_full_dataset", "data_vanilla"]
columns = ["molecule_name", "calculated_property", "reference_property", "error", "calRelError"]

fig, axes = plt.subplots(2, 1, figsize=(16, 12), sharey=True)
axes = axes.flatten()

# Custom legend for SAScore tiers
tier_handles = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="red", markersize=8, label="tier 0/1"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="black", markersize=8, label="tier 2"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="green", markersize=8, label="tier 3"),
]

# Legend for targets (from violinplot hue)
target_palette = sns.color_palette("Set2", 4)
target_labels = ["target0", "target1", "target2", "target3"]
target_handles = [
    Line2D([0], [0], marker="s", color="w", markerfacecolor=target_palette[i], markersize=10, label=target_labels[i])
    for i in range(4)
]

for exp_idx, exp in enumerate(experiments):
    exp_path = DATA_ROOT / exp
    data = []

    target_folders = [
        (exp_path / "evaluation/target0", "target0"),
        (exp_path / "evaluation/target1", "target1"),
        (exp_path / "evaluation/target2", "target2"),
        (exp_path / "evaluation/target3", "target3"),
    ]

    for folder, target_name in target_folders:
        for model_name in os.listdir(folder):
            model_path = os.path.join(folder, model_name)
            if not os.path.isdir(model_path):
                continue

            csv_path = os.path.join(model_path, "results/eMBD.csv")
            if not os.path.exists(csv_path):
                continue

            df = pd.read_csv(csv_path, header=None, names=columns, sep="\t")
            df["model"] = model_name
            df["target"] = target_name
            df = df.drop_duplicates(subset="molecule_name", keep="first")
            df["relative_error"] = 100 * ((df["calculated_property"] - df["reference_property"]) / df["reference_property"])

            df_top40 = df.loc[df["relative_error"].abs().sort_values(ascending=top).index].head(n)

            sas_path = os.path.join(model_path, "results/sascore.txt")
            if os.path.exists(sas_path):
                df_sas = pd.read_csv(sas_path, header=None, sep=r"\s+")
                df_sas = df_sas[[0, 2]]
                df_sas.columns = ["molecule_name", "tier"]
                df_sas["molecule_name"] = df_sas["molecule_name"].astype(str).str.strip()
                df_sas["tier"] = df_sas["tier"].astype(str).str.strip()
            else:
                df_sas = pd.DataFrame(columns=["molecule_name", "tier"])

            df_top40 = df_top40.merge(df_sas, on="molecule_name", how="left")

            def tier_to_color(t):
                if t in ["tier_0", "tier_1"]:
                    return "red"
                elif t == "tier_2":
                    return "black"
                else:
                    return "green"

            df_top40["tier_color"] = df_top40["tier"].apply(tier_to_color)
            data.append(df_top40)

    df_all = pd.concat(data, ignore_index=True)
    df_all["model"] = pd.Categorical(df_all["model"], categories=sorted(df_all["model"].unique()), ordered=True)

    ax = axes[exp_idx]
    sns.violinplot(data=df_all, x="model", y="relative_error", hue="target", palette="Set2", split=True, ax=ax, inner="quartile")
    sns.stripplot(data=df_all, x="model", y="relative_error",
                  hue="tier_color", palette={"red": "red", "black": "black", "green": "green"},
                  dodge=False, size=4, jitter=True, marker="o", alpha=0.8, ax=ax)

    ax.axhline(0, color="red", linestyle="--")
    ax.axhline(5, color="blue", linestyle="--")
    ax.axhline(-5, color="blue", linestyle="--")

    ax.set_title(exp)
    ax.set_xlabel("Model")
    if exp_idx == 0:
        ax.set_ylabel("Relative Error (%)")
    ax.tick_params(axis="x", rotation=45)
    if ax.legend_ is not None:
        ax.legend_.remove()

plt.legend(handles=target_handles + tier_handles, loc="lower right")
plt.tight_layout()
plt.show()
